In [1]:
import requests
import polars as pl
import plotly.express as px

**Load the data**

In [2]:
import requests
import time

all_data = []
page = 0

while True:
    url = f"https://api.tvmaze.com/shows?page={page}"
    response = requests.get(url)

    if response.status_code == 404:
        print("No more pages. Data collection finished.")
        break

    response.raise_for_status()

    page_data = response.json()
    all_data.extend(page_data)

    print(f"Page {page}: {len(page_data)} records | Total: {len(all_data)}")

    page += 1
    time.sleep(0.5)

data = all_data

print("\nFinal number of records:", len(data))

Page 0: 240 records | Total: 240
Page 1: 245 records | Total: 485
Page 2: 242 records | Total: 727
Page 3: 243 records | Total: 970
Page 4: 238 records | Total: 1208
Page 5: 233 records | Total: 1441
Page 6: 234 records | Total: 1675
Page 7: 241 records | Total: 1916
Page 8: 232 records | Total: 2148
Page 9: 231 records | Total: 2379
Page 10: 229 records | Total: 2608
Page 11: 228 records | Total: 2836
Page 12: 226 records | Total: 3062
Page 13: 235 records | Total: 3297
Page 14: 228 records | Total: 3525
Page 15: 229 records | Total: 3754
Page 16: 232 records | Total: 3986
Page 17: 242 records | Total: 4228
Page 18: 241 records | Total: 4469
Page 19: 235 records | Total: 4704
Page 20: 241 records | Total: 4945
Page 21: 241 records | Total: 5186
Page 22: 235 records | Total: 5421
Page 23: 243 records | Total: 5664
Page 24: 236 records | Total: 5900
Page 25: 228 records | Total: 6128
Page 26: 232 records | Total: 6360
Page 27: 229 records | Total: 6589
Page 28: 231 records | Total: 6820

In [3]:
data[0]

{'id': 1,
 'url': 'https://www.tvmaze.com/shows/1/under-the-dome',
 'name': 'Under the Dome',
 'type': 'Scripted',
 'language': 'English',
 'genres': ['Drama', 'Science-Fiction', 'Thriller'],
 'status': 'Ended',
 'runtime': 60,
 'averageRuntime': 60,
 'premiered': '2013-06-24',
 'ended': '2015-09-10',
 'officialSite': 'http://www.cbs.com/shows/under-the-dome/',
 'schedule': {'time': '22:00', 'days': ['Thursday']},
 'rating': {'average': 6.6},
 'weight': 100,
 'network': {'id': 2,
  'name': 'CBS',
  'country': {'name': 'United States',
   'code': 'US',
   'timezone': 'America/New_York'},
  'officialSite': 'https://www.cbs.com/'},
 'webChannel': None,
 'dvdCountry': None,
 'externals': {'tvrage': 25988, 'thetvdb': 264492, 'imdb': 'tt1553656'},
 'image': {'medium': 'https://static.tvmaze.com/uploads/images/medium_portrait/610/1525272.jpg',
  'original': 'https://static.tvmaze.com/uploads/images/original_untouched/610/1525272.jpg'},
 'summary': "<p><b>Under the Dome</b> is the story of a s

In [4]:
df = pl.DataFrame({
    "id": [x.get("id") for x in data],
    "name": [x.get("name") for x in data],
    "type": [x.get("type") for x in data],
    "language": [x.get("language") for x in data],
    "genres": [", ".join(x.get("genres", [])) for x in data],
    "status": [x.get("status") for x in data],
    "runtime": [x.get("runtime") for x in data],
    "average_runtime": [x.get("averageRuntime") for x in data],
    "premiered": [x.get("premiered") for x in data],
    "ended": [x.get("ended") for x in data],
    "rating": [x.get("rating", {}).get("average") for x in data],
    "weight": [x.get("weight") for x in data],
    "network": [
        x.get("network", {}).get("name")
        if x.get("network") else None
        for x in data
    ],
    "country": [
        x.get("network", {}).get("country", {}).get("name")
        if x.get("network") and x.get("network").get("country")
        else None
        for x in data
    ]
})

df.head()

id,name,type,language,genres,status,runtime,average_runtime,premiered,ended,rating,weight,network,country
i64,str,str,str,str,str,i64,i64,str,str,f64,i64,str,str
1,"""Under the Dome""","""Scripted""","""English""","""Drama, Science-Fiction, Thrill…","""Ended""",60,60,"""2013-06-24""","""2015-09-10""",6.6,100,"""CBS""","""United States"""
2,"""Person of Interest""","""Scripted""","""English""","""Action, Crime, Science-Fiction""","""Ended""",60,60,"""2011-09-22""","""2016-06-21""",8.8,100,"""CBS""","""United States"""
3,"""Bitten""","""Scripted""","""English""","""Drama, Horror, Romance""","""Ended""",60,60,"""2014-01-11""","""2016-04-15""",7.4,99,"""CTV Sci-Fi Channel""","""Canada"""
4,"""Arrow""","""Scripted""","""English""","""Drama, Action, Science-Fiction""","""Ended""",60,60,"""2012-10-10""","""2020-01-28""",7.4,99,"""The CW""","""United States"""
5,"""True Detective""","""Scripted""","""English""","""Drama, Crime, Thriller""","""Running""",60,63,"""2014-01-12""",null,8.1,100,"""HBO""","""United States"""


In [5]:
print("Shape:", df.shape)
print("Columns:", df.columns)

Shape: (89736, 14)
Columns: ['id', 'name', 'type', 'language', 'genres', 'status', 'runtime', 'average_runtime', 'premiered', 'ended', 'rating', 'weight', 'network', 'country']


In [6]:
df.head(10)

id,name,type,language,genres,status,runtime,average_runtime,premiered,ended,rating,weight,network,country
i64,str,str,str,str,str,i64,i64,str,str,f64,i64,str,str
1,"""Under the Dome""","""Scripted""","""English""","""Drama, Science-Fiction, Thrill…","""Ended""",60,60,"""2013-06-24""","""2015-09-10""",6.6,100,"""CBS""","""United States"""
2,"""Person of Interest""","""Scripted""","""English""","""Action, Crime, Science-Fiction""","""Ended""",60,60,"""2011-09-22""","""2016-06-21""",8.8,100,"""CBS""","""United States"""
3,"""Bitten""","""Scripted""","""English""","""Drama, Horror, Romance""","""Ended""",60,60,"""2014-01-11""","""2016-04-15""",7.4,99,"""CTV Sci-Fi Channel""","""Canada"""
4,"""Arrow""","""Scripted""","""English""","""Drama, Action, Science-Fiction""","""Ended""",60,60,"""2012-10-10""","""2020-01-28""",7.4,99,"""The CW""","""United States"""
5,"""True Detective""","""Scripted""","""English""","""Drama, Crime, Thriller""","""Running""",60,63,"""2014-01-12""",null,8.1,100,"""HBO""","""United States"""
6,"""The 100""","""Scripted""","""English""","""Action, Adventure, Science-Fic…","""Ended""",60,60,"""2014-03-19""","""2020-09-30""",7.7,100,"""The CW""","""United States"""
7,"""Homeland""","""Scripted""","""English""","""Drama, Thriller, Espionage""","""Ended""",60,60,"""2011-10-02""","""2020-04-26""",8.2,99,"""Paramount+ with Showtime""","""United States"""
8,"""Glee""","""Scripted""","""English""","""Drama, Music, Romance""","""Ended""",60,60,"""2009-05-19""","""2015-03-20""",6.6,98,"""FOX""","""United States"""
9,"""Revenge""","""Scripted""","""English""","""Drama, Thriller, Mystery""","""Ended""",60,60,"""2011-09-21""","""2015-05-10""",7.8,97,"""ABC""","""United States"""


In [7]:
df.schema

Schema([('id', Int64),
        ('name', String),
        ('type', String),
        ('language', String),
        ('genres', String),
        ('status', String),
        ('runtime', Int64),
        ('average_runtime', Int64),
        ('premiered', String),
        ('ended', String),
        ('rating', Float64),
        ('weight', Int64),
        ('network', String),
        ('country', String)])

In [8]:
df.null_count()

id,name,type,language,genres,status,runtime,average_runtime,premiered,ended,rating,weight,network,country
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,2803,0,0,25144,9366,3459,22609,78010,0,23927,23927


In [9]:
print("rows before removing duplicates:", df.height)

df = df.unique(subset=["id"])

print("Rows after removing duplicates:", df.height)

rows before removing duplicates: 89736
Rows after removing duplicates: 89736


**Data type conversion**

In [10]:
df = df.with_columns([
    pl.col("premiered").str.to_date(strict=False),
    pl.col("ended").str.to_date(strict=False)
])

df.schema

Schema([('id', Int64),
        ('name', String),
        ('type', String),
        ('language', String),
        ('genres', String),
        ('status', String),
        ('runtime', Int64),
        ('average_runtime', Int64),
        ('premiered', Date),
        ('ended', Date),
        ('rating', Float64),
        ('weight', Int64),
        ('network', String),
        ('country', String)])

In [11]:
df.select([
    pl.col("rating").is_null().sum().alias("missing_rating"),
    pl.col("runtime").is_null().sum().alias("missing_runtime"),
    pl.col("average_runtime").is_null().sum().alias("missing_average_runtime")
])

missing_rating,missing_runtime,missing_average_runtime
u32,u32,u32
78010,25144,9366


**Data Cleaning**

In [12]:
df_clean = df.filter(pl.col("rating").is_not_null())

In [13]:
print("Original rows:", df.height)
print("Clean analysis rows:", df_clean.height)

Original rows: 89736
Clean analysis rows: 11726


In [14]:
df_clean.head(10)

id,name,type,language,genres,status,runtime,average_runtime,premiered,ended,rating,weight,network,country
i64,str,str,str,str,str,i64,i64,date,date,f64,i64,str,str
1775,"""Schitt's Creek""","""Scripted""","""English""","""Comedy""","""Ended""",30,30,2015-01-13,2020-04-07,7.6,99,"""CBC""","""Canada"""
89912,"""Dirty Business""","""Scripted""","""English""","""Drama""","""Ended""",65,65,2026-02-23,2026-02-25,8.5,93,"""Channel 4""","""United Kingdom"""
18627,"""Strong Woman Do Bong Soon""","""Scripted""","""Korean""","""Comedy, Action, Fantasy""","""Ended""",60,60,2017-02-24,2017-04-15,8.0,61,"""jTBC""","""Korea, Republic of"""
7116,"""Nightwing: The Series""","""Scripted""","""English""","""Action, Fantasy""","""Ended""",null,10,2014-09-29,2014-10-27,6.5,91,null,null
12401,"""Moon Embracing the Sun""","""Scripted""","""Korean""","""Drama, Fantasy, Romance""","""Ended""",70,70,2012-01-04,2012-03-15,7.4,80,"""MBC""","""Korea, Republic of"""
64736,"""The Bequeathed""","""Scripted""","""Korean""","""Crime, Mystery""","""Ended""",null,47,2024-01-19,2024-01-19,5.2,87,null,null
69003,"""Knokke Off""","""Scripted""","""Dutch""","""Drama""","""Running""",null,32,2023-05-12,null,6.8,83,null,null
79236,"""Starting 5""","""Documentary""","""English""","""Sports""","""Ended""",null,48,2024-10-09,2025-10-16,6.5,74,null,null
191,"""Halt and Catch Fire""","""Scripted""","""English""","""Drama""","""Ended""",60,60,2014-06-01,2017-10-14,8.1,98,"""AMC""","""United States"""


In [15]:
print("Final shape:", df_clean.shape)
print("\nMissing values:")
print(df_clean.null_count())

Final shape: (11726, 14)

Missing values:
shape: (1, 14)
┌─────┬──────┬──────┬──────────┬───┬────────┬────────┬─────────┬─────────┐
│ id  ┆ name ┆ type ┆ language ┆ … ┆ rating ┆ weight ┆ network ┆ country │
│ --- ┆ ---  ┆ ---  ┆ ---      ┆   ┆ ---    ┆ ---    ┆ ---     ┆ ---     │
│ u32 ┆ u32  ┆ u32  ┆ u32      ┆   ┆ u32    ┆ u32    ┆ u32     ┆ u32     │
╞═════╪══════╪══════╪══════════╪═══╪════════╪════════╪═════════╪═════════╡
│ 0   ┆ 0    ┆ 0    ┆ 69       ┆ … ┆ 0      ┆ 0      ┆ 3536    ┆ 3536    │
└─────┴──────┴──────┴──────────┴───┴────────┴────────┴─────────┴─────────┘


In [16]:
print("Status Code:", response.status_code)
print(type(data))
print("Number of records:", len(data))

Status Code: 404
<class 'list'>
Number of records: 89736


**Filling null values**

In [17]:
df_clean = df_clean.with_columns(pl.col("runtime").fill_null(pl.col("average_runtime")).alias("runtime"))

df_clean.select([pl.col("runtime").is_null().sum().alias("missing_runtime")])

missing_runtime
u32
57


In [18]:
df_clean = df_clean.with_columns(pl.col("premiered").dt.year().alias("premiere_year"))

df_clean.select(["name","premiered","premiere_year","rating","runtime"]).head(10)

name,premiered,premiere_year,rating,runtime
str,date,i32,f64,i64
"""Schitt's Creek""",2015-01-13,2015,7.6,30
"""Dirty Business""",2026-02-23,2026,8.5,65
"""Strong Woman Do Bong Soon""",2017-02-24,2017,8.0,60
"""Nightwing: The Series""",2014-09-29,2014,6.5,10
"""Moon Embracing the Sun""",2012-01-04,2012,7.4,70
"""The Bequeathed""",2024-01-19,2024,5.2,47
"""Knokke Off""",2023-05-12,2023,6.8,32
"""Starting 5""",2024-10-09,2024,6.5,48
"""Halt and Catch Fire""",2014-06-01,2014,8.1,60


In [19]:
df_clean.describe()

statistic,id,name,type,language,genres,status,runtime,average_runtime,premiered,ended,rating,weight,network,country,premiere_year
str,f64,str,str,str,str,str,f64,f64,str,str,f64,f64,str,str,f64
"""count""",11726.0,"""11726""","""11726""","""11657""","""11726""","""11726""",11669.0,11601.0,"""11723""","""9293""",11726.0,11726.0,"""8190""","""8190""",11723.0
"""null_count""",0.0,"""0""","""0""","""69""","""0""","""0""",57.0,125.0,"""3""","""2433""",0.0,0.0,"""3536""","""3536""",3.0
"""mean""",32674.215419,null,null,null,null,null,46.161968,46.173606,"""2013-08-04 18:30:48.059370""","""2014-09-03 14:31:18.833530""",6.710549,80.528398,null,null,2013.107908
"""std""",26380.541024,null,null,null,null,null,22.975408,22.662818,null,null,1.13345,17.143394,null,null,12.642474
"""min""",1.0,"""#1 Happy Family USA""","""Animation""","""Afrikaans""","""""","""Ended""",1.0,1.0,"""1929-05-01""","""1950-10-26""",1.0,8.0,"""1+1""","""Argentina""",1929.0
"""25%""",5637.0,null,null,null,null,null,30.0,30.0,"""2010-09-26""","""2012-03-22""",6.2,71.0,null,null,2010.0
"""50%""",32792.0,null,null,null,null,null,48.0,48.0,"""2017-08-06""","""2018-07-15""",7.0,86.0,null,null,2017.0
"""75%""",54241.0,null,null,null,null,null,60.0,60.0,"""2021-08-06""","""2022-01-14""",7.5,94.0,null,null,2021.0
"""max""",93948.0,"""Юность""","""Variety""","""Zulu""","""Western""","""To Be Determined""",219.0,219.0,"""2026-09-04""","""2026-09-07""",9.6,100.0,"""Ю""","""United States""",2026.0


In [20]:
rating_stats = df_clean.select([
    pl.col("rating").count().alias("Number of Shows"),
    pl.col("rating").mean().alias("Average Rating"),
    pl.col("rating").median().alias("Median Rating"),
    pl.col("rating").min().alias("Minimum Rating"),
    pl.col("rating").max().alias("Maximum Rating"),
    pl.col("rating").std().alias("Standard Deviation")
])

rating_stats

Number of Shows,Average Rating,Median Rating,Minimum Rating,Maximum Rating,Standard Deviation
u32,f64,f64,f64,f64,f64
11726,6.710549,7.0,1.0,9.6,1.13345


In [21]:
runtime_stats = df_clean.select([
    pl.col("runtime").count().alias("Number of Shows"),
    pl.col("runtime").mean().alias("Average Runtime"),
    pl.col("runtime").median().alias("Median Runtime"),
    pl.col("runtime").min().alias("Minimum Runtime"),
    pl.col("runtime").max().alias("Maximum Runtime"),
    pl.col("runtime").std().alias("Standard Deviation")
])

runtime_stats

Number of Shows,Average Runtime,Median Runtime,Minimum Runtime,Maximum Runtime,Standard Deviation
u32,f64,f64,i64,i64,f64
11669,46.161968,48.0,1,219,22.975408


In [22]:
genre_df = (df_clean.with_columns(pl.col("genres").str.split(", ")).explode("genres"))

genre_counts = (genre_df.group_by("genres").agg(pl.len().alias("Number of Shows")).sort("Number of Shows", descending=True))

genre_counts

genres,Number of Shows
str,u32
"""Drama""",4922
"""Comedy""",3470
"""Crime""",1856
"""Action""",1678
"""Romance""",1382
…,…
"""War""",133
"""Espionage""",110
"""Western""",72


In [23]:
genre_counts_clean = genre_counts.filter(pl.col("genres") != "")
genre_counts_clean

genres,Number of Shows
str,u32
"""Drama""",4922
"""Comedy""",3470
"""Crime""",1856
"""Action""",1678
"""Romance""",1382
…,…
"""War""",133
"""Espionage""",110
"""Western""",72


**relationship between TV show genres and their ratings**

In [24]:
genre_rating = (genre_df.filter(pl.col("genres") != "").group_by("genres").agg([pl.len().alias("Number of Shows"),pl.col("rating").mean().alias("Average Rating")]).sort("Average Rating", descending=True))
genre_rating

genres,Number of Shows,Average Rating
str,u32,f64
"""Nature""",146,7.441096
"""Western""",72,7.426389
"""War""",133,7.293985
"""Travel""",135,7.095556
"""Espionage""",110,7.074545
…,…,…
"""Family""",569,6.621441
"""Sports""",169,6.497633
"""Music""",233,6.319742


**Number of Shows**

In [25]:
type_counts = (df_clean.group_by("type").agg(pl.len().alias("Number of Shows")).sort("Number of Shows", descending=True))

type_counts

type,Number of Shows
str,u32
"""Scripted""",6885
"""Animation""",2525
"""Documentary""",969
"""Reality""",864
"""Game Show""",147
…,…
"""Talk Show""",111
"""News""",37
"""Sports""",31


In [26]:
df.select([
    "runtime",
    "average_runtime",
    "rating",
    "weight"
]).describe()

statistic,runtime,average_runtime,rating,weight
str,f64,f64,f64,f64
"""count""",64592.0,80370.0,11726.0,89736.0
"""null_count""",25144.0,9366.0,78010.0,0.0
"""mean""",48.686246,46.358504,6.710549,45.788446
"""std""",27.029835,27.28945,1.13345,26.259863
"""min""",1.0,1.0,1.0,0.0
"""25%""",30.0,30.0,6.2,25.0
"""50%""",50.0,45.0,7.0,45.0
"""75%""",60.0,60.0,7.5,65.0
"""max""",540.0,540.0,9.6,100.0


In [27]:
numeric_columns = [
    "runtime",
    "average_runtime",
    "rating",
    "weight"
]

for column in numeric_columns:

    values = (
        df
        .select(pl.col(column))
        .drop_nulls()
        .to_series()
    )

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = values.filter(
        (values < lower_bound) |
        (values > upper_bound)
    )

    print("=" * 60)
    print(f"Column: {column}")
    print(f"Q1: {q1:.2f}")
    print(f"Q3: {q3:.2f}")
    print(f"IQR: {iqr:.2f}")
    print(f"Lower Bound: {lower_bound:.2f}")
    print(f"Upper Bound: {upper_bound:.2f}")
    print(f"Minimum: {values.min()}")
    print(f"Maximum: {values.max()}")
    print(f"Outliers: {len(outliers)}")
    print(f"Outlier %: {(len(outliers) / len(values) * 100):.2f}%")

Column: runtime
Q1: 30.00
Q3: 60.00
IQR: 30.00
Lower Bound: -15.00
Upper Bound: 105.00
Minimum: 1
Maximum: 540
Outliers: 2497
Outlier %: 3.87%
Column: average_runtime
Q1: 30.00
Q3: 60.00
IQR: 30.00
Lower Bound: -15.00
Upper Bound: 105.00
Minimum: 1
Maximum: 540
Outliers: 2780
Outlier %: 3.46%
Column: rating
Q1: 6.20
Q3: 7.50
IQR: 1.30
Lower Bound: 4.25
Upper Bound: 9.45
Minimum: 1.0
Maximum: 9.6
Outliers: 406
Outlier %: 3.46%
Column: weight
Q1: 25.00
Q3: 65.00
IQR: 40.00
Lower Bound: -35.00
Upper Bound: 125.00
Minimum: 0
Maximum: 100
Outliers: 0
Outlier %: 0.00%


In [28]:
for column in numeric_columns:

    values = (
        df
        .select(pl.col(column))
        .drop_nulls()
        .to_series()
    )

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df.filter(
        (pl.col(column) < lower_bound) |
        (pl.col(column) > upper_bound)
    )

    print("\n" + "=" * 70)
    print(f"OUTLIERS FOR: {column}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    print(f"Number of outliers: {outliers.height}")

    print(
        outliers.select([
            "id",
            "name",
            "runtime",
            "average_runtime",
            "rating",
            "weight",
            "premiered"
        ]).head(20)
    )


OUTLIERS FOR: runtime
Lower bound: -15.00
Upper bound: 105.00
Number of outliers: 2497
shape: (20, 7)
┌───────┬───────────────────────────────┬─────────┬─────────────────┬────────┬────────┬────────────┐
│ id    ┆ name                          ┆ runtime ┆ average_runtime ┆ rating ┆ weight ┆ premiered  │
│ ---   ┆ ---                           ┆ ---     ┆ ---             ┆ ---    ┆ ---    ┆ ---        │
│ i64   ┆ str                           ┆ i64     ┆ i64             ┆ f64    ┆ i64    ┆ date       │
╞═══════╪═══════════════════════════════╪═════════╪═════════════════╪════════╪════════╪════════════╡
│ 64411 ┆ Russell Quirk                 ┆ 120     ┆ 120             ┆ null   ┆ 7      ┆ 2022-10-02 │
│ 72939 ┆ Ne Gemiler Yaktım             ┆ 130     ┆ 130             ┆ null   ┆ 14     ┆ 2023-12-04 │
│ 69312 ┆ Ya Çok Seversen               ┆ 130     ┆ 130             ┆ null   ┆ 74     ┆ 2023-07-06 │
│ 90028 ┆ Doktor: Başka Hayatta         ┆ 140     ┆ 140             ┆ null   ┆ 7      ┆ 2

In [29]:
fig = px.box(
    df.select(["runtime"]).drop_nulls(),
    y="runtime",
    title="Runtime Box Plot and Outliers"
)

fig.show()

In [30]:
rating_data = (
    df
    .select(["rating"])
    .drop_nulls()
)

fig = px.histogram(
    rating_data,
    x="rating",
    nbins=30,
    title="Distribution of TV Show Ratings"
)

fig.show()

In [31]:
fig = px.box(
    df.select(["rating"]).drop_nulls(),
    y="rating",
    title="Rating Box Plot and Outliers"
)

fig.show()

In [32]:
fig = px.box(
    df.select(["weight"]).drop_nulls(),
    y="weight",
    title="Weight Box Plot and Outliers"
)

fig.show()

In [33]:
fig = px.box(
    df.select(["average_runtime"]).drop_nulls(),
    y="average_runtime",
    title="Average Runtime Box Plot"
)

fig.show()

In [34]:
fig = px.histogram(
    df.select(["average_runtime"]).drop_nulls(),
    x="average_runtime",
    nbins=40,
    title="Distribution of Average Runtime"
)

fig.show()

In [35]:
df.select("premiered").describe()

statistic,premiered
str,str
"""count""","""86277"""
"""null_count""","""3459"""
"""mean""","""2013-11-10 04:10:03.358948"""
"""std""",null
"""min""","""1922-09-10"""
"""25%""","""2010-08-10"""
"""50%""","""2017-07-02"""
"""75%""","""2022-04-29"""
"""max""","""2027-01-22"""


In [36]:
print("Earliest premiere:")
print(df.select(pl.col("premiered").min()))

print("\nLatest premiere:")
print(df.select(pl.col("premiered").max()))

Earliest premiere:
shape: (1, 1)
┌────────────┐
│ premiered  │
│ ---        │
│ date       │
╞════════════╡
│ 1922-09-10 │
└────────────┘

Latest premiere:
shape: (1, 1)
┌────────────┐
│ premiered  │
│ ---        │
│ date       │
╞════════════╡
│ 2027-01-22 │
└────────────┘


In [37]:
future_or_late = df.filter(
    pl.col("premiered") > pl.date(2026, 5, 31)
)

print("Shows after May 2026:", future_or_late.height)

future_or_late.select([
    "id",
    "name",
    "premiered",
    "status",
    "rating",
    "runtime"
]).head(20)

Shows after May 2026: 1099


id,name,premiered,status,rating,runtime
i64,str,date,str,f64,i64
94036,"""Japan's Wild Side""",2026-09-06,"""Running""",null,60
91588,"""House of Stassi""",2026-07-30,"""To Be Determined""",null,null
93023,"""Rewriting the Fate of Jiang Hu…",2026-08-09,"""Ended""",null,null
93204,"""Come Dine with Me: Celebrity P…",2026-07-23,"""Running""",null,null
93255,"""Yun Bin Luan""",2026-07-29,"""Ended""",null,null
…,…,…,…,…,…
92829,"""Анатомия чувств""",2026-07-03,"""Ended""",null,null
93952,"""Forbidden Love In Beijing""",2026-06-12,"""Running""",null,null
83305,"""Blossoms of Power""",2026-07-09,"""Ended""",null,null


In [38]:
df_year = df.with_columns(
    pl.col("premiered")
    .dt.year()
    .alias("premiere_year")
)

year_counts = (
    df_year
    .group_by("premiere_year")
    .agg(pl.len().alias("count"))
    .sort("premiere_year")
)

year_counts

premiere_year,count
i32,u32
null,3459
1922,1
1929,1
1930,1
1933,1
…,…
2023,5222
2024,5044
2025,4994


In [39]:
import plotly.express as px

fig = px.line(
    year_counts.to_pandas(),
    x="premiere_year",
    y="count",
    title="TV Shows by Premiere Year"
)

fig.show()

**Analysis**

In [40]:

import polars as pl

ml_df = df.clone()


ml_df = ml_df.with_columns([
    pl.col("runtime").cast(pl.Float64, strict=False),
    pl.col("rating").cast(pl.Float64, strict=False),
    pl.col("weight").cast(pl.Float64, strict=False)
])


ml_df = ml_df.filter(
    pl.col("premiered").is_not_null() &
    (pl.col("premiered") >= pl.date(1990, 1, 1)) &
    (pl.col("premiered") <= pl.date(2026, 5, 31))
)

ml_df = ml_df.filter(
    pl.col("runtime").is_not_null() &
    (pl.col("runtime") >= 0) &
    (pl.col("runtime") <= 180)
)

ml_df = ml_df.unique(
    subset=["id"],
    keep="first"
)

if ml_df.height >= 60000:

    ml_df = ml_df.sample(
        n=60000,
        seed=42
    )

    print("60,000 unique shows successfully selected.")

else:

    print(
        f"Only {ml_df.height} unique shows are available "
        "with the current filters."
    )

ml_df = ml_df.select([
    "id",
    "name",
    "genres",
    "runtime",
    "premiered",
    "rating",
    "weight",
    "language",
    "type",
    "status",
    "network",
    "country"
])

print()

print("FINAL SHOW ML DATASET")

print("Rows:", ml_df.height)
print("Columns:", ml_df.width)

print(
    "Runtime:",
    ml_df["runtime"].min(),
    "→",
    ml_df["runtime"].max()
)

print(
    "Premiere:",
    ml_df["premiered"].min(),
    "→",
    ml_df["premiered"].max()
)

print(
    "Rating:",
    ml_df["rating"].min(),
    "→",
    ml_df["rating"].max()
)

print(
    "Weight:",
    ml_df["weight"].min(),
    "→",
    ml_df["weight"].max()
)

print(
    "Unique IDs:",
    ml_df["id"].n_unique()
)


Only 57769 unique shows are available with the current filters.

FINAL SHOW ML DATASET
Rows: 57769
Columns: 12
Runtime: 1.0 → 180.0
Premiere: 1990-01-01 → 2026-05-31
Rating: 1.0 → 9.5
Weight: 0.0 → 100.0
Unique IDs: 57769


In [41]:


ml_df = ml_df.with_columns(
    pl.when(pl.col("rating") >= 5)
    .then(1)
    .otherwise(0)
    .alias("label")
)

print("Total Shows:", ml_df.height)

print("\nLow Rated (< 5):")
print(ml_df.filter(pl.col("label") == 0).height)

print("\nHigh Rated (>= 5):")
print(ml_df.filter(pl.col("label") == 1).height)

print("\nSample:")
print(
    ml_df.select([
        "name",
        "genres",
        "runtime",
        "premiered",
        "rating",
        "weight",
        "label"
    ]).head(10)
)

Total Shows: 57769

Low Rated (< 5):
50871

High Rated (>= 5):
6898

Sample:
shape: (10, 7)
┌───────────────────────────────┬────────────────┬─────────┬────────────┬────────┬────────┬───────┐
│ name                          ┆ genres         ┆ runtime ┆ premiered  ┆ rating ┆ weight ┆ label │
│ ---                           ┆ ---            ┆ ---     ┆ ---        ┆ ---    ┆ ---    ┆ ---   │
│ str                           ┆ str            ┆ f64     ┆ date       ┆ f64    ┆ f64    ┆ i32   │
╞═══════════════════════════════╪════════════════╪═════════╪════════════╪════════╪════════╪═══════╡
│ Jacht op Guido & Emma         ┆                ┆ 5.0     ┆ 2016-06-30 ┆ null   ┆ 18.0   ┆ 0     │
│ Baby Beauty Queens            ┆                ┆ 30.0    ┆ 2010-07-27 ┆ null   ┆ 23.0   ┆ 0     │
│ Синяя роза                    ┆ Drama          ┆ 45.0    ┆ 2017-08-06 ┆ null   ┆ 28.0   ┆ 0     │
│ World's Greatest Head to Head ┆ Sports         ┆ 30.0    ┆ 2018-05-28 ┆ null   ┆ 45.0   ┆ 0     │
│ Aa, Lo

In [42]:

ml_df = ml_df.filter(
    pl.col("rating").is_not_null()
)

ml_df = ml_df.with_columns(
    pl.col("premiered")
    .dt.year()
    .cast(pl.Float64)
    .alias("premiere_year")
)

genres = [
    "Drama",
    "Comedy",
    "Crime",
    "Action",
    "Adventure",
    "Romance",
    "Thriller",
    "Mystery",
    "Family",
    "Science-Fiction"
]

for g in genres:
    ml_df = ml_df.with_columns(
        pl.col("genres")
        .fill_null("")
        .str.contains(g, literal=True)
        .cast(pl.Int32)
        .alias("genre_" + g.lower().replace("-", "_"))
    )

# Recreate target correctly
ml_df = ml_df.with_columns(
    pl.when(pl.col("rating") >= 5)
    .then(1)
    .otherwise(0)
    .alias("label")
)

print("Rows:", ml_df.height)

print("\nLow Rated (< 5):",
      ml_df.filter(pl.col("label") == 0).height)

print("High Rated (>= 5):",
      ml_df.filter(pl.col("label") == 1).height)

print("\nFeatures:")
print([
    "runtime",
    "weight",
    "premiere_year"
] + ["genre_" + g.lower().replace("-", "_") for g in genres])

print("\nSample:")
print(
    ml_df.select([
        "name",
        "genres",
        "runtime",
        "premiere_year",
        "rating",
        "weight",
        "label"
    ]).head(10)
)

Rows: 7570

Low Rated (< 5): 672
High Rated (>= 5): 6898

Features:
['runtime', 'weight', 'premiere_year', 'genre_drama', 'genre_comedy', 'genre_crime', 'genre_action', 'genre_adventure', 'genre_romance', 'genre_thriller', 'genre_mystery', 'genre_family', 'genre_science_fiction']

Sample:
shape: (10, 7)
┌───────────────────────┬──────────────────────┬─────────┬───────────────┬────────┬────────┬───────┐
│ name                  ┆ genres               ┆ runtime ┆ premiere_year ┆ rating ┆ weight ┆ label │
│ ---                   ┆ ---                  ┆ ---     ┆ ---           ┆ ---    ┆ ---    ┆ ---   │
│ str                   ┆ str                  ┆ f64     ┆ f64           ┆ f64    ┆ f64    ┆ i32   │
╞═══════════════════════╪══════════════════════╪═════════╪═══════════════╪════════╪════════╪═══════╡
│ Dead of Summer        ┆ Drama, Horror        ┆ 60.0    ┆ 2016.0        ┆ 6.3    ┆ 91.0   ┆ 1     │
│ Aşk Laftan Anlamaz    ┆ Drama, Romance       ┆ 110.0   ┆ 2016.0        ┆ 7.9    ┆ 82.0 

In [43]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler

spark = SparkSession.builder \
    .appName("TVmaze_ML_Analysis") \
    .getOrCreate()

print("Spark version:", spark.version)

spark_ml_df = spark.createDataFrame(
    ml_df.to_pandas()
)

feature_columns = [
    "runtime",
    "weight",
    "premiere_year",
    "genre_drama",
    "genre_comedy",
    "genre_crime",
    "genre_action",
    "genre_adventure",
    "genre_romance",
    "genre_thriller",
    "genre_mystery",
    "genre_family",
    "genre_science_fiction"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

spark_ml_df = assembler.transform(spark_ml_df)

spark_ml_df = spark_ml_df.select(
    "features",
    "label"
)

print("SPARK ML DATA")

print("Rows:", spark_ml_df.count())
print("Features:", len(feature_columns))

print("\nLabel Distribution:")
spark_ml_df.groupBy("label") \
    .count() \
    .orderBy("label") \
    .show()

Spark version: 4.0.4
SPARK ML DATA
Rows: 7570
Features: 13

Label Distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0|  672|
|    1| 6898|
+-----+-----+



In [44]:

train_df, test_df = spark_ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

print("\nOriginal Training Distribution:")
train_df.groupBy("label").count().orderBy("label").show()


low_train = train_df.filter("label = 0")
high_train = train_df.filter("label = 1")

low_count = low_train.count()
high_count = high_train.count()

low_balanced = low_train.sample(
    withReplacement=True,
    fraction=high_count / low_count,
    seed=42
)

balanced_train_df = high_train.unionByName(low_balanced)

print("\nBalanced Training Distribution:")
balanced_train_df.groupBy("label").count().orderBy("label").show()

print("Balanced Training rows:", balanced_train_df.count())

Training rows: 6126
Testing rows: 1444

Original Training Distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0|  546|
|    1| 5580|
+-----+-----+


Balanced Training Distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0| 5579|
|    1| 5580|
+-----+-----+

Balanced Training rows: 11159


In [45]:

from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


best_dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=20,
    minInstancesPerNode=1,
    seed=42
)

best_dt_model = best_dt.fit(balanced_train_df)

# Predictions on TEST data only
best_predictions = best_dt_model.transform(test_df)


evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

accuracy = evaluator.evaluate(
    best_predictions,
    {evaluator.metricName: "accuracy"}
)

precision = evaluator.evaluate(
    best_predictions,
    {evaluator.metricName: "weightedPrecision"}
)

recall = evaluator.evaluate(
    best_predictions,
    {evaluator.metricName: "weightedRecall"}
)

f1 = evaluator.evaluate(
    best_predictions,
    {evaluator.metricName: "f1"}
)


print("BEST DECISION TREE - TEST RESULTS")


print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")

print("CONFUSION MATRIX")

best_predictions.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
).show()

BEST DECISION TREE - TEST RESULTS
Accuracy  : 80.96%
Precision : 85.71%
Recall    : 80.96%
F1-Score  : 83.11%
CONFUSION MATRIX
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|   33|
|    0|       1.0|   93|
|    1|       0.0|  182|
|    1|       1.0| 1136|
+-----+----------+-----+



In [46]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

dash_df = ml_df.to_pandas()

dash_df["label_name"] = dash_df["label"].map({
    0: "Low Rated",
    1: "High Rated"
})

dash_df["runtime_group"] = pd.cut(
    dash_df["runtime"],
    bins=[0, 60, 90, 120, 150, 180],
    labels=[
        "45–60",
        "61–90",
        "91–120",
        "121–150",
        "151–180"
    ]
)

dash_df["premiere_year"] = (
    pd.to_datetime(dash_df["premiered"])
    .dt.year
)

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[
        "High vs Low Rated by Runtime",
        "High vs Low Rated by Genre",
        "High vs Low Rated by Weight",
        "Average Rating by Genre",
        "High vs Low Rated by Premiere Year"
    ],
    specs=[
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "box"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "xy"}]
    ]
)

runtime_data = (
    dash_df.groupby(
        ["runtime_group", "label_name"],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

for label in ["Low Rated", "High Rated"]:

    d = runtime_data[
        runtime_data["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=d["runtime_group"].astype(str),
            y=d["count"],
            name=label,
            legendgroup=label
        ),
        row=1,
        col=1
    )


genre_df = dash_df[
    ["genres", "label_name"]
].copy()

genre_df["genres"] = genre_df["genres"].fillna("Unknown")

genre_df["genre"] = genre_df["genres"].str.split(", ")

genre_df = genre_df.explode("genre")

genre_counts = (
    genre_df.groupby(
        ["genre", "label_name"]
    )
    .size()
    .reset_index(name="count")
)

top_genres = (
    genre_df["genre"]
    .value_counts()
    .head(10)
    .index
)

genre_counts = genre_counts[
    genre_counts["genre"].isin(top_genres)
]

for label in ["Low Rated", "High Rated"]:

    d = genre_counts[
        genre_counts["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=d["genre"],
            y=d["count"],
            name=label,
            legendgroup=label,
            showlegend=False
        ),
        row=1,
        col=2
    )


for label in ["Low Rated", "High Rated"]:

    d = dash_df[
        dash_df["label_name"] == label
    ]

    fig.add_trace(
        go.Box(
            y=d["weight"],
            name=label,
            legendgroup=label,
            showlegend=False
        ),
        row=2,
        col=1
    )


rating_genre = (
    genre_df
    .groupby(["genre", "label_name"])["label_name"]
    .size()
    .reset_index(name="count")
)


genre_rating = genre_df.merge(
    dash_df[["genres", "rating", "label_name"]],
    on=["genres", "label_name"],
    how="left"
)

genre_rating = (
    genre_rating
    .groupby(["genre", "label_name"])["rating"]
    .mean()
    .reset_index()
)

genre_rating = genre_rating[
    genre_rating["genre"].isin(top_genres)
]

for label in ["Low Rated", "High Rated"]:

    d = genre_rating[
        genre_rating["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=d["genre"],
            y=d["rating"],
            name=label,
            legendgroup=label,
            showlegend=False
        ),
        row=2,
        col=2
    )

year_label = (
    dash_df.groupby(
        ["premiere_year", "label_name"]
    )
    .size()
    .reset_index(name="count")
)

for label in ["Low Rated", "High Rated"]:

    d = year_label[
        year_label["label_name"] == label
    ]

    fig.add_trace(
        go.Scatter(
            x=d["premiere_year"],
            y=d["count"],
            mode="lines+markers",
            name=label,
            legendgroup=label,
            showlegend=False
        ),
        row=3,
        col=1
    )

# ------------------------------------------
# Dashboard Layout
# ------------------------------------------

fig.update_layout(
    height=1100,
    width=1200,
    title="Dashboard 1 — TV Show Rating & Characteristics",
    barmode="group",
    template="plotly_white"
)

fig.update_xaxes(title_text="Runtime (minutes)", row=1, col=1)
fig.update_yaxes(title_text="Number of Shows", row=1, col=1)

fig.update_xaxes(title_text="Genre", row=1, col=2)
fig.update_yaxes(title_text="Number of Shows", row=1, col=2)

fig.update_yaxes(title_text="Weight", row=2, col=1)

fig.update_xaxes(title_text="Genre", row=2, col=2)
fig.update_yaxes(title_text="Average Rating", row=2, col=2)

fig.update_xaxes(title_text="Premiere Year", row=3, col=1)
fig.update_yaxes(title_text="Number of Shows", row=3, col=1)

fig.show()

In [48]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

d2 = ml_df.to_pandas()

d2["label_name"] = d2["label"].map({
    0: "Low Rated",
    1: "High Rated"
})

d2["premiere_year"] = pd.to_datetime(
    d2["premiered"]
).dt.year

for col in ["language", "country", "status", "network"]:
    d2[col] = d2[col].fillna("Unknown").astype(str)


# Create dashboard with 5 visualisations
fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[
        "High vs Low Rated by Language",
        "High vs Low Rated by Country",
        "High vs Low Rated by Production Network",
        "High vs Low Rated by Show Status",
        "",
        "Average Weight by Premiere Year"
    ]
)


# -------------------------
# 1. LANGUAGE
# -------------------------

language = (
    d2.groupby(["language", "label_name"])
    .size()
    .reset_index(name="count")
)

top_languages = (
    d2["language"]
    .value_counts()
    .head(10)
    .index
)

language = language[
    language["language"].isin(top_languages)
]

for label in ["Low Rated", "High Rated"]:

    x = language[
        language["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=x["language"],
            y=x["count"],
            name=label
        ),
        row=1,
        col=1
    )


# -------------------------
# 2. COUNTRY
# -------------------------

country = (
    d2.groupby(["country", "label_name"])
    .size()
    .reset_index(name="count")
)

top_countries = (
    d2["country"]
    .value_counts()
    .head(10)
    .index
)

country = country[
    country["country"].isin(top_countries)
]

for label in ["Low Rated", "High Rated"]:

    x = country[
        country["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=x["country"],
            y=x["count"],
            name=label,
            showlegend=False
        ),
        row=1,
        col=2
    )


# -------------------------
# 3. PRODUCTION NETWORK
# -------------------------

network = (
    d2.groupby(["network", "label_name"])
    .size()
    .reset_index(name="count")
)

top_networks = (
    d2["network"]
    .value_counts()
    .head(10)
    .index
)

network = network[
    network["network"].isin(top_networks)
]

for label in ["Low Rated", "High Rated"]:

    x = network[
        network["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=x["network"],
            y=x["count"],
            name=label,
            showlegend=False
        ),
        row=2,
        col=1
    )


# -------------------------
# 4. SHOW STATUS
# -------------------------

status = (
    d2.groupby(["status", "label_name"])
    .size()
    .reset_index(name="count")
)

for label in ["Low Rated", "High Rated"]:

    x = status[
        status["label_name"] == label
    ]

    fig.add_trace(
        go.Bar(
            x=x["status"],
            y=x["count"],
            name=label,
            showlegend=False
        ),
        row=2,
        col=2
    )


# -------------------------
# 5. AVERAGE WEIGHT BY YEAR
# -------------------------

year_weight = (
    d2.groupby(
        ["premiere_year", "label_name"]
    )["weight"]
    .mean()
    .reset_index()
)

for label in ["Low Rated", "High Rated"]:

    x = year_weight[
        year_weight["label_name"] == label
    ]

    fig.add_trace(
        go.Scatter(
            x=x["premiere_year"],
            y=x["weight"],
            mode="lines+markers",
            name=label,
            showlegend=False
        ),
        row=3,
        col=2
    )


# -------------------------
# DASHBOARD LAYOUT
# -------------------------

fig.update_layout(
    title="Dashboard 2 — Market & Production Strategy",
    height=1100,
    width=1250,
    barmode="group",
    template="plotly_white"
)


# -------------------------
# AXIS LABELS
# -------------------------

fig.update_xaxes(
    title_text="Language",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Number of Shows",
    row=1,
    col=1
)


fig.update_xaxes(
    title_text="Country",
    row=1,
    col=2
)

fig.update_yaxes(
    title_text="Number of Shows",
    row=1,
    col=2
)


fig.update_xaxes(
    title_text="Production Network",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Number of Shows",
    row=2,
    col=1
)


fig.update_xaxes(
    title_text="Status",
    row=2,
    col=2
)

fig.update_yaxes(
    title_text="Number of Shows",
    row=2,
    col=2
)


fig.update_xaxes(
    title_text="Premiere Year",
    row=3,
    col=2
)

fig.update_yaxes(
    title_text="Average Weight",
    row=3,
    col=2
)


fig.show()